## Shower/Track Classifier GNN

For more information: https://github.com/a2ronmeade/TrackShowerGNN/wiki

#### **1.1: Imports**

In [ ]:
# file / data utilities
import sys, os, glob, time, gc
import pickle, uproot, csv
import importlib.metadata as ilmd
from pathlib import Path
from tqdm.auto import tqdm

# numerical / data
import numpy as np
import awkward as ak
import pandas as pd

# scikit-learn (splitting + evaluation metrics)
from sklearn.model_selection import train_test_split
from scipy.spatial import Delaunay, QhullError
from multiprocessing import Pool
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,confusion_matrix, roc_curve, roc_auc_score, precision_recall_curve, f1_score

# GNN Architecture
import torch
import torch_geometric
import torch.multiprocessing
import torch.nn as nn
from torch.nn import Linear, Sequential, Tanh, LayerNorm, BCEWithLogitsLoss
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, global_max_pool, global_add_pool, MessagePassing
from torch_geometric.utils import softmax as pyg_softmax
torch.multiprocessing.set_sharing_strategy('file_system')

# optimization
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# plotting
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# use GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


#### **1.2: Version Check**

In [ ]:
print(f"Python version:            {sys.version.split()[0]}")
print(f"PyTorch version:           {torch.__version__}")
print(f"PyTorch Geometric version: {torch_geometric.__version__}")
print(f"NumPy version:             {np.__version__}")
print(f"Awkward Array version:     {ak.__version__}")
print(f"Pandas version:            {pd.__version__}")
print(f"uproot version:            {uproot.__version__}")
print(f"scikit-learn version:      {ilmd.version('scikit-learn')}")
print(f"Matplotlib version:        {plt.matplotlib.__version__}")
print(f"Seaborn version:           {sns.__version__}")
print(f"IPython version:           {ilmd.version('ipython')}")

print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name:  {torch.cuda.get_device_name(0)}")


#### **2.1: Configuration and Setup**

* See wiki page for more information on variables: https://github.com/a2ronmeade/TrackShowerGNN/wiki/5.-Configuration-Variables

In [ ]:
# basic utilities
NICKNAME        = 'FinalModel_withFVcut'
SHOW_PLOTS      = False
DATA_DIR        = 'FCData'
BRANCH          = 'LArRecoND'
FIELDS_NEEDED   = ['event','clusterId','isShower','recoHitX','recoHitY','recoHitZ','recoHitT0','recoHitE','recoHitClusterId','mcPDG','sliceId','recoHitSliceId',
                 'length1','length2','length3','energy','n3DHits','nUHits','nVHits','nWHits','nuVtxX', 'nuVtxY', 'nuVtxZ']

# particle info
SHOWER_PDGS     = {11, -11, 22}
PDG_CATEGORIES = {
    'Muon':     {13, -13},
    'Proton':   {2212},
    'Pion':     {211, -211, 111},
    'Kaon':     {321, -321, 130, 310, 311},
    'Electron': {11, -11},
    'Photon':   {22},
}
CATEGORY_COLORS = {
    'Muon':     'darkorange',
    'Proton':   'firebrick',
    'Pion':     'goldenrod',
    'Kaon':     'purple',
    'Electron': 'steelblue',
    'Photon':   'seagreen',
}

# data loading
LOAD_FROM_DISK = True
CACHE_PATH      = 'clusters_mini.pkl'

# filtering
MIN_HITS        = 10
APPLY_FV        = 'all' # 'all', 'none' or 'val and test'
CONTAINMENT_DIST= 5.0
MAX_COORD       = 1000.0

# optimization
RUN_OPTIMIZATION    = False
N_OPTUNA_TRIALS     = 30
EPOCHS_PER_TRIAL    = 12    
OPTIMIZE_METRIC     = 'loss'

# architecture
FALLBACK_K  = 16
DECISION_THRESHOLD   = 0.5
DROPOUT_P            = 0.1 
ATTN_HEADS           = 4  
BATCH_SIZE           = 512

# training
EPOCHS         = 80
PATIENCE       = 15
CHECKPOINT_DIR = Path(f'{NICKNAME}/checkpoints')
LOG_PATH       = Path(f'{NICKNAME}/training_log.csv')
PLOTS_PATH     = Path(f'{NICKNAME}/plots')



# other:
DELAUNAY_USE_DISK = False
DELAUNAY_PATH = Path(f'delaunay_edges_minhits{MIN_HITS}.pkl')

#### **2.2: Load ROOT files (only if not using cache)**

In [ ]:
if not LOAD_FROM_DISK:
    # find all ROOT files
    all_files = glob.glob(os.path.join(DATA_DIR, '*.root'))
    print(f"Total ROOT files found: {len(all_files)}")

    # sort files by type using filename prefix
    events_files = sorted([f for f in all_files if os.path.basename(f).startswith('EventHierarchy')])
    reco_files   = sorted([f for f in all_files if os.path.basename(f).startswith('LArRecoND')])
    mc_files     = sorted([f for f in all_files if os.path.basename(f).startswith('MCHierarchy')])

    print(f"Events files:   {len(events_files)}")
    print(f"LArRecoND files: {len(reco_files)}")
    print(f"MC files:       {len(mc_files)}")
    assert len(events_files) == len(reco_files) == len(mc_files), \
        "Mismatch in file counts across tree types!"

    # load all LArRecoND files into a list of awkward arrays
    print("\nLoading LArRecoND files...")
    raw_data = []
    for f in reco_files:
        tree = uproot.open(f)[BRANCH]
        arrays = tree.arrays(FIELDS_NEEDED, library='ak')
        raw_data.append(arrays)
        print(f"  Loaded: {os.path.basename(f)}")

    # Concatenate into one large awkward array
    data = ak.concatenate(raw_data, axis=0)


#### **3.1: Preprocessing**

In [ ]:
# loads the pickle file from disk
if LOAD_FROM_DISK:
    print(f"Loading preprocessed clusters from {CACHE_PATH} ...")
    with open(CACHE_PATH, 'rb') as f:
        cache = pickle.load(f)
    clusters = cache['clusters']
    print(f"Loaded {len(clusters):,} clusters.")

    required_cluster_fields = ['length1', 'length2', 'length3', 'energy', 'n3DHits', 'nUHits', 'nVHits', 'nWHits',
                                'nuVtxX', 'nuVtxY', 'nuVtxZ', 'rms', 'linearity', 'planarity', 'scattering',
                                'event_idx', 'sliceId', 'clusterId']
    missing = [f for f in required_cluster_fields if f not in clusters[0]]
    if missing:
        raise KeyError(f"Saved clusters are missing field(s) {missing}. Set LOAD_FROM_DISK = False in Section 2.1 and rerun to regenerate {CACHE_PATH} with the new fields.")

# builds the pickle file and saves to disk
else:
    def compute_shape_ratios(l1, l2, l3, eps=1e-6):
        return {
            'linearity':  (l1 - l2) / (l1 + eps),
            'planarity':  (l2 - l3) / (l1 + eps),
            'scattering': l3 / (l1 + eps),
        }

    clusters = []
    for event_idx, entry in enumerate(data):
        # hit level features
        cluster_ids     = np.array(entry['clusterId'])
        slice_ids       = np.array(entry['sliceId'])
        pdgs            = np.array(entry['mcPDG'])
        hit_X           = np.array(entry['recoHitX'])
        hit_Y           = np.array(entry['recoHitY'])
        hit_Z           = np.array(entry['recoHitZ'])
        hit_T0          = np.array(entry['recoHitT0'])
        hit_E           = np.array(entry['recoHitE'])
        hit_cluster_ids = np.array(entry['recoHitClusterId'])
        hit_slice_ids   = np.array(entry['recoHitSliceId'])

        # cluster level features
        length1_arr = np.array(entry['length1'])
        length2_arr = np.array(entry['length2'])
        length3_arr = np.array(entry['length3'])
        energy_arr  = np.array(entry['energy'])
        n3DHits_arr = np.array(entry['n3DHits'])
        nUHits_arr  = np.array(entry['nUHits'])
        nVHits_arr  = np.array(entry['nVHits'])
        nWHits_arr  = np.array(entry['nWHits'])
        nu_vtx_x_arr = np.array(entry['nuVtxX'])
        nu_vtx_y_arr = np.array(entry['nuVtxY'])
        nu_vtx_z_arr = np.array(entry['nuVtxZ'])

        cluster_pdg = {}
        cluster_globals = {}
        for i in range(len(cluster_ids)):
            key = (slice_ids[i], cluster_ids[i])
            cluster_pdg[key] = pdgs[i]
            cluster_globals[key] = {
                'length1': float(length1_arr[i]), 'length2': float(length2_arr[i]), 'length3': float(length3_arr[i]),
                'energy': float(energy_arr[i]),
                'n3DHits': float(n3DHits_arr[i]), 'nUHits': float(nUHits_arr[i]),
                'nVHits': float(nVHits_arr[i]), 'nWHits': float(nWHits_arr[i]),
                'nuVtxX': float(nu_vtx_x_arr[i]), 'nuVtxY': float(nu_vtx_y_arr[i]), 'nuVtxZ': float(nu_vtx_z_arr[i]),
            }

        # group hits by sliceId and clusterId
        hit_groups = {}
        for i in range(len(hit_X)):
            key = (hit_slice_ids[i], hit_cluster_ids[i])
            if key not in hit_groups:
                hit_groups[key] = {'X': [], 'Y': [], 'Z': [], 'T0': [], 'E': []}
            hit_groups[key]['X'].append(hit_X[i])
            hit_groups[key]['Y'].append(hit_Y[i])
            hit_groups[key]['Z'].append(hit_Z[i])
            hit_groups[key]['T0'].append(hit_T0[i])
            hit_groups[key]['E'].append(hit_E[i])

        # build one dict per cluster
        for key, hits in hit_groups.items():
            if key not in cluster_pdg:
                continue

            pdg = cluster_pdg[key]

            # skip unmatched clusters
            if pdg == 0:
                continue

            n_hits = len(hits['X'])
            if n_hits < MIN_HITS:
                continue

            label = 0 if pdg in SHOWER_PDGS else 1
            g = cluster_globals[key]

            # compute RMS using distance from the cluster centroid
            hit_x_arr = np.asarray(hits['X'], dtype=np.float64)
            hit_y_arr = np.asarray(hits['Y'], dtype=np.float64)
            hit_z_arr = np.asarray(hits['Z'], dtype=np.float64)
            centroid_x, centroid_y, centroid_z = hit_x_arr.mean(), hit_y_arr.mean(), hit_z_arr.mean()
            rms = float(np.sqrt(np.mean((hit_x_arr - centroid_x) ** 2 + (hit_y_arr - centroid_y) ** 2 + (hit_z_arr - centroid_z) ** 2)))

            #linearity, planarity, scattering are computed here
            shape_ratios = compute_shape_ratios(g['length1'], g['length2'], g['length3'])

            # Store raw (unnormalized) hits, plus event, slice and cluster identifiers
            clusters.append({
                'X':      np.array(hits['X'],  dtype=np.float32),
                'Y':      np.array(hits['Y'],  dtype=np.float32),
                'Z':      np.array(hits['Z'],  dtype=np.float32),
                'T0':     np.array(hits['T0'], dtype=np.float32),
                'E':      np.array(hits['E'],  dtype=np.float32),
                'label':  label,
                'pdg':    pdg,
                'n_hits': n_hits,
                'rms':    rms,
                'event_idx': event_idx,
                'sliceId':   int(key[0]),
                'clusterId': int(key[1]),
                **shape_ratios,
                **g,
            })

    n_showers = sum(1 for c in clusters if c['label'] == 0)
    n_tracks  = sum(1 for c in clusters if c['label'] == 1)
    total     = n_showers + n_tracks

    print(f"Total clusters after filtering: {len(clusters):,}")
    print(f"Showers (label=0): {n_showers:,} ({100*n_showers/total:.1f}%)")
    print(f"Tracks  (label=1): {n_tracks:,}  ({100*n_tracks/total:.1f}%)")

    # write to disk
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump({'clusters': clusters}, f)
    print(f"Saved {len(clusters):,} raw clusters to {CACHE_PATH}")

# prints out info about the clusters and the track/shower balance
n_showers = sum(1 for c in clusters if c['label'] == 0)
n_tracks  = sum(1 for c in clusters if c['label'] == 1)
total     = n_showers + n_tracks
print(f"\nClass balance:  Showers: {n_showers:,} ({100*n_showers/total:.1f}%) Tracks: {n_tracks:,} ({100*n_tracks/total:.1f}%)")

#### **3.2: Detector Containment Filtering**

In [ ]:
# defines the detector geometry, format is: (x_center, half_x, y_center, half_y, z_center, half_z) for each of the TPCs, in cm.
TPC_BOUNDS = np.array([
    ( -323.7130, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 0
    ( -276.2870, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 1
    ( -323.7130, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 2
    ( -276.2870, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 3
    ( -323.7130, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 4
    ( -276.2870, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 5
    ( -323.7130, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 6
    ( -276.2870, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 7
    ( -323.7130, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 8
    ( -276.2870, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 9
    ( -223.7130, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 10
    ( -176.2870, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 11
    ( -223.7130, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 12
    ( -176.2870, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 13
    ( -223.7130, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 14
    ( -176.2870, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 15
    ( -223.7130, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 16
    ( -176.2870, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 17
    ( -223.7130, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 18
    ( -176.2870, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 19
    ( -123.7130, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 20
    (  -76.2870, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 21
    ( -123.7130, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 22
    (  -76.2870, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 23
    ( -123.7130, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 24
    (  -76.2870, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 25
    ( -123.7130, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 26
    (  -76.2870, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 27
    ( -123.7130, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 28
    (  -76.2870, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 29
    (  -23.7130, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 30
    (   23.7130, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 31
    (  -23.7130, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 32
    (   23.7130, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 33
    (  -23.7130, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 34
    (   23.7130, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 35
    (  -23.7130, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 36
    (   23.7130, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 37
    (  -23.7130, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 38
    (   23.7130, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 39
    (   76.2870, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 40
    (  123.7130, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 41
    (   76.2870, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 42
    (  123.7130, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 43
    (   76.2870, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 44
    (  123.7130, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 45
    (   76.2870, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 46
    (  123.7130, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 47
    (   76.2870, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 48
    (  123.7130, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 49
    (  176.2870, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 50
    (  223.7130, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 51
    (  176.2870, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 52
    (  223.7130, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 53
    (  176.2870, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 54
    (  223.7130, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 55
    (  176.2870, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 56
    (  223.7130, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 57
    (  176.2870, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 58
    (  223.7130, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 59
    (  276.2870, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 60
    (  323.7130, 23.3955,   -66.8713, 149.7990,   465.7560, 47.8318),  # TPC 61
    (  276.2870, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 62
    (  323.7130, 23.3955,   -66.8713, 149.7990,   565.7560, 47.8318),  # TPC 63
    (  276.2870, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 64
    (  323.7130, 23.3955,   -66.8713, 149.7990,   665.7560, 47.8318),  # TPC 65
    (  276.2870, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 66
    (  323.7130, 23.3955,   -66.8713, 149.7990,   765.7560, 47.8318),  # TPC 67
    (  276.2870, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 68
    (  323.7130, 23.3955,   -66.8713, 149.7990,   865.7560, 47.8318),  # TPC 69
], dtype=np.float64)

def tpc_membership(x, y, z, bounds=TPC_BOUNDS):
    # index of the TPC (raw box, no margin) containing each point, or -1 if none.
    x = np.asarray(x); y = np.asarray(y); z = np.asarray(z)
    tpc_id = np.full(x.shape, -1, dtype=np.int64)
    for j in range(bounds.shape[0]):
        xc, xh, yc, yh, zc, zh = bounds[j]
        unassigned = tpc_id == -1
        if not unassigned.any():
            break
        mask = unassigned & (np.abs(x - xc) <= xh) & (np.abs(y - yc) <= yh) & (np.abs(z - zc) <= zh)
        tpc_id[mask] = j
    return tpc_id

def is_contained(x, y, z, dist, bounds=TPC_BOUNDS):
    # true where a point lies at least some distance inside every face of some TPC's box.
    x = np.asarray(x); y = np.asarray(y); z = np.asarray(z)
    contained = np.zeros(x.shape, dtype=bool)
    for j in range(bounds.shape[0]):
        xc, xh, yc, yh, zc, zh = bounds[j]
        mask = ((np.abs(x - xc) <= xh - dist) & (np.abs(y - yc) <= yh - dist) & (np.abs(z - zc) <= zh - dist))
        contained |= mask
    return contained

def cluster_passes_containment(c, dist, bounds=TPC_BOUNDS):
    # true where the whole cluster passes the containment criteria (hits + vertex)
    if not is_contained(c['X'], c['Y'], c['Z'], dist, bounds).all():
        return False
    tpc_ids = tpc_membership(c['X'], c['Y'], c['Z'], bounds)
    if (tpc_ids == -1).any() or len(np.unique(tpc_ids)) > 1:
        return False
    vtx_ok = is_contained(np.array([c['nuVtxX']]), np.array([c['nuVtxY']]), np.array([c['nuVtxZ']]), dist, bounds)[0]
    return bool(vtx_ok)

print(f"Containment helpers defined ({TPC_BOUNDS.shape[0]} TPCs). "
      f"Cut is applied per-split in Section 4.1 according to APPLY_FV={APPLY_FV!r}.")


#### **3.3: Feature Preparation**

In [ ]:
labels = [c['label'] for c in clusters]

# split into 80/10/10 train/val/test
train_idx, temp_idx = train_test_split(range(len(clusters)), test_size=0.20, stratify=labels, random_state=42)
temp_labels = [labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(temp_idx,test_size=0.50, stratify=temp_labels, random_state=42)

train_clusters = [clusters[i] for i in train_idx]
val_clusters   = [clusters[i] for i in val_idx]
test_clusters  = [clusters[i] for i in test_idx]

# applies the containment criteria defined in section 3.2
def apply_containment(cluster_list, idx_list, split_name):
    n_before = len(cluster_list)
    kept_clusters, kept_idx = [], []
    for c, i in tqdm(zip(cluster_list, idx_list), total=n_before,
                      desc=f"Containment filter ({split_name})", unit="cluster"):
        if cluster_passes_containment(c, CONTAINMENT_DIST):
            kept_clusters.append(c)
            kept_idx.append(i)
    print(f"Containment filter (CONTAINMENT_DIST={CONTAINMENT_DIST} cm, {split_name}): kept {len(kept_clusters):,} / {n_before:,} clusters ")
    return kept_clusters, kept_idx

# actual application here
if APPLY_FV == 'all':
    train_clusters, train_idx = apply_containment(train_clusters, train_idx, 'train')
    val_clusters, val_idx     = apply_containment(val_clusters,   val_idx,   'val')
    test_clusters, test_idx   = apply_containment(test_clusters,  test_idx,  'test')
elif APPLY_FV == 'val and test':
    val_clusters, val_idx   = apply_containment(val_clusters,  val_idx,  'val')
    test_clusters, test_idx = apply_containment(test_clusters, test_idx, 'test')
elif APPLY_FV == 'none':
    pass

print(f"Split — Train: {len(train_clusters):,}  "
      f"Val: {len(val_clusters):,}  "
      f"Test: {len(test_clusters):,}")

# coordinate range sanity check, train set only. compare against MAX_COORD
all_X_raw = np.concatenate([c['X'] for c in train_clusters])
all_Y_raw = np.concatenate([c['Y'] for c in train_clusters])
all_Z_raw = np.concatenate([c['Z'] for c in train_clusters])

print(f"\nRaw coordinate range (train set) -- compare against MAX_COORD={MAX_COORD}:")
print(f"  X: [{all_X_raw.min():.1f}, {all_X_raw.max():.1f}]")
print(f"  Y: [{all_Y_raw.min():.1f}, {all_Y_raw.max():.1f}]")
print(f"  Z: [{all_Z_raw.min():.1f}, {all_Z_raw.max():.1f}]")
max_abs_coord = max(np.abs(all_X_raw).max(), np.abs(all_Y_raw).max(), np.abs(all_Z_raw).max())
print(f"  Largest absolute coordinate seen: {max_abs_coord:.1f}")

# normalization statistics for T0 and E, train set only (X/Y/Z use MAX_COORD scale instead)
all_T0 = np.concatenate([c['T0'] for c in train_clusters])
all_E  = np.concatenate([c['E']  for c in train_clusters])
means = {'T0': float(all_T0.mean()), 'E':  float(all_E.mean()),}
stds = {'T0': float(all_T0.std()),'E':  float(all_E.std()),}

# global cluster-level feature normalization
GLOBAL_ZSCORE_FEATS = ['length1', 'length2', 'length3', 'rms', 'energy','n3DHits', 'nUHits', 'nVHits', 'nWHits']
for feat in GLOBAL_ZSCORE_FEATS:
    arr = np.array([c[feat] for c in train_clusters], dtype=np.float64)
    means[feat] = float(arr.mean())
    stds[feat]  = float(arr.std()) if arr.std() > 0 else 1.0

print("\nZ-score normalization stats (train set only):")
for k in means:
    print(f"  {k}: mean={means[k]:.4f}  std={stds[k]:.4f}")

def normalize_cluster(c, means, stds, max_coord):
    out = {
        **c,
        'X':  c['X'] / max_coord,
        'Y':  c['Y'] / max_coord,
        'Z':  c['Z'] / max_coord,
        'T0': (c['T0'] - means['T0']) / stds['T0'],
        'E':  (c['E']  - means['E'])  / stds['E'],
    }
    for feat in GLOBAL_ZSCORE_FEATS:
        out[feat] = (c[feat] - means[feat]) / stds[feat]
    return out

train_clusters = [normalize_cluster(c, means, stds, MAX_COORD) for c in train_clusters]
val_clusters   = [normalize_cluster(c, means, stds, MAX_COORD) for c in val_clusters]
test_clusters  = [normalize_cluster(c, means, stds, MAX_COORD) for c in test_clusters]

print("\nNormalization applied (X/Y/Z: fixed-scale, T0/E/global-feats: z-score, "
      "shape-ratios: unchanged).")

# pos_weight for logits loss, from train set only
n_train_showers = sum(1 for c in train_clusters if c['label'] == 0)
n_train_tracks  = sum(1 for c in train_clusters if c['label'] == 1)
pos_weight = torch.tensor([n_train_showers / n_train_tracks], dtype=torch.float32)
print(f"\npos_weight (train set): {pos_weight.item():.4f}")


#### **4.1: Graph Construction**

In [ ]:
# edge construction for a single cluster
def delaunay_edges_for_cluster(xyz):

    n = xyz.shape[0]
    if n < 4:
        # if there are too few points to tetrahedralize in 3D then just fully connect them.
        idx = np.arange(n)
        src, dst = np.meshgrid(idx, idx, indexing='ij')
        mask = src != dst
        return src[mask].astype(np.int64), dst[mask].astype(np.int64)
    try:
        tri = Delaunay(xyz, qhull_options='QJ')
        simplices = tri.simplices  # (M, 4) tetrahedra, each a 4-tuple of point indices
        pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]  # 6 edges/tetrahedron
        e = np.concatenate([simplices[:, [i, j]] for i, j in pairs], axis=0)
        e = np.concatenate([e, e[:, ::-1]], axis=0)  # symmetrize (undirected -> both directions)
        e = np.unique(e, axis=0)
        return e[:, 0].astype(np.int64), e[:, 1].astype(np.int64)
    except QhullError:
        # degenerate points fall back to a small kNN
        k = min(FALLBACK_K, n - 1)
        d = np.linalg.norm(xyz[:, None, :] - xyz[None, :, :], axis=-1)
        np.fill_diagonal(d, np.inf)
        idx = np.argsort(d, axis=1)[:, :k]
        src = np.repeat(np.arange(n), k)
        dst = idx.flatten()
        return src.astype(np.int64), dst.astype(np.int64)


def compute_edges_for_split(cluster_list, desc):
    xyz_list = [np.stack([c['X'], c['Y'], c['Z']], axis=1) for c in cluster_list]

    results = []
    with Pool(processes=max(1, (os.cpu_count() or 2) - 1)) as pool:
        for src, dst in tqdm(pool.imap(delaunay_edges_for_cluster, xyz_list, chunksize=64), total=len(xyz_list), desc=desc):
            results.append((src, dst))

    offsets = np.zeros(len(results) + 1, dtype=np.int64)
    for i, (src, _) in enumerate(results):
        offsets[i + 1] = offsets[i] + len(src)

    edge_src = (np.concatenate([r[0] for r in results]) if results else np.array([], dtype=np.int64))
    edge_dst = (np.concatenate([r[1] for r in results]) if results else np.array([], dtype=np.int64))

    total_edges = len(edge_src)
    avg_degree = total_edges / max(1, len(cluster_list))
    print(f"  {desc}: {len(cluster_list):,} clusters -> {total_edges:,} directed edges "
          f"(avg degree {avg_degree:.1f})")

    return edge_src, edge_dst, offsets


# delauney triangulation is computed here
if DELAUNAY_USE_DISK and DELAUNAY_PATH.exists():
    print(f"Loading cached Delaunay edges from {DELAUNAY_PATH} ...")
    with open(DELAUNAY_PATH, 'rb') as f:
        edge_cache = pickle.load(f)
else:
    print("Computing Delaunay triangulation edges for this run (into RAM, no disk cache)..."
          if not DELAUNAY_USE_DISK else
          "No cache found -- computing Delaunay triangulation edges (will be cached)...")
    t_start = time.time()
    edge_cache = {
        'train': compute_edges_for_split(train_clusters, 'Delaunay (train)'),
        'val':   compute_edges_for_split(val_clusters,   'Delaunay (val)'),
        'test':  compute_edges_for_split(test_clusters,  'Delaunay (test)'),
    }
    print(f"Done in {time.time() - t_start:.1f}s.")
    if DELAUNAY_USE_DISK:
        with open(DELAUNAY_PATH, 'wb') as f:
            pickle.dump(edge_cache, f)
        print(f"Cached to {DELAUNAY_PATH}. Future runs will load this instantly instead of recomputing.")

print(f"\nAvg node degree (train, Delaunay): ~{len(edge_cache['train'][0]) / max(1, len(train_clusters)):.1f}")


#### **4.2: Flatten clusters + DataLoader**

In [ ]:
# global (cluster-level) feature vector attached to each graph
GLOBAL_FEATS = ['length1', 'length2', 'length3', 'rms', 'energy', 'n3DHits', 'nUHits', 'nVHits', 'nWHits', 'linearity', 'planarity', 'scattering']

# flatten cluster dicts into contiguous numpy arrays
def clusters_to_arrays(cluster_list, edges=None):
    offsets = np.zeros(len(cluster_list) + 1, dtype=np.int64)
    for i, c in enumerate(cluster_list):
        offsets[i + 1] = offsets[i] + len(c['X'])

    n_hits_total = offsets[-1]
    X  = np.empty(n_hits_total, dtype=np.float32)
    Y  = np.empty(n_hits_total, dtype=np.float32)
    Z  = np.empty(n_hits_total, dtype=np.float32)
    T0 = np.empty(n_hits_total, dtype=np.float32)
    E  = np.empty(n_hits_total, dtype=np.float32)
    labels = np.empty(len(cluster_list), dtype=np.float32)
    global_feats = np.empty((len(cluster_list), len(GLOBAL_FEATS)), dtype=np.float32)

    for i, c in enumerate(cluster_list):
        s, e = offsets[i], offsets[i + 1]
        X[s:e], Y[s:e], Z[s:e], T0[s:e], E[s:e] = c['X'], c['Y'], c['Z'], c['T0'], c['E']
        labels[i] = c['label']
        global_feats[i] = [c[f] for f in GLOBAL_FEATS]

    out = {'X': X, 'Y': Y, 'Z': Z, 'T0': T0, 'E': E, 'labels': labels,
           'offsets': offsets, 'global_feats': global_feats}

    # precomputed graph connectivity, stored the same way as the node arrays above: flat arrays + an offsets index
    if edges is not None:
        edge_src, edge_dst, edge_offsets = edges
        out['edge_src'] = edge_src
        out['edge_dst'] = edge_dst
        out['edge_offsets'] = edge_offsets
    return out


print("Flattening train clusters...")
train_arrays = clusters_to_arrays(train_clusters, edges=edge_cache['train'] if edge_cache else None)
print("Flattening val clusters...")
val_arrays   = clusters_to_arrays(val_clusters,   edges=edge_cache['val']   if edge_cache else None)
print("Flattening test clusters...")
test_arrays  = clusters_to_arrays(test_clusters,  edges=edge_cache['test']  if edge_cache else None)


class ClusterDataset(Dataset):
    def __init__(self, arrays):
        super().__init__()
        self.a = arrays

    def len(self):
        return len(self.a['offsets']) - 1

    def get(self, idx):
        s, e = self.a['offsets'][idx], self.a['offsets'][idx + 1]
        node_features = torch.from_numpy(
            np.stack([self.a['X'][s:e], self.a['Y'][s:e], self.a['Z'][s:e], self.a['T0'][s:e], self.a['E'][s:e]], axis=1)).float()
        
        pos = node_features[:, :4]   # X, Y, Z, T0 -- pos_xyz feeds RelativeEdgeConv's relative-position term
        y = torch.tensor([float(self.a['labels'][idx])], dtype=torch.float32)
        u = torch.from_numpy(self.a['global_feats'][idx]).float().unsqueeze(0)
        data = Data(x=node_features, pos=pos, y=y, u=u)

        if 'edge_offsets' in self.a:
            es, ee = self.a['edge_offsets'][idx], self.a['edge_offsets'][idx + 1]
            data.edge_index = torch.from_numpy(np.stack([self.a['edge_src'][es:ee], self.a['edge_dst'][es:ee]])).long()
        return data


train_dataset = ClusterDataset(train_arrays)
val_dataset   = ClusterDataset(val_arrays)
test_dataset  = ClusterDataset(test_arrays)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders ready — Train batches: {len(train_loader)}  "
      f"Val batches: {len(val_loader)}  Test batches: {len(test_loader)}")
print(f"Global feature vector length: {len(GLOBAL_FEATS)} -> {GLOBAL_FEATS}")


#### **4.3: Fourier Feature Encoding**

In [ ]:
class FourierFeatureEncoder(nn.Module):
    def __init__(self, num_freqs=6, include_input=True, input_scale=1.0):
        super().__init__()
        self.num_freqs = num_freqs
        self.include_input = include_input
        self.input_scale = input_scale

        freq_bands = 2.0 ** torch.arange(num_freqs, dtype=torch.float32)
        self.register_buffer('freq_bands', freq_bands)

    def out_dim(self, in_dim):
        per_channel = (1 if self.include_input else 0) + 2 * self.num_freqs
        return in_dim * per_channel

    def forward(self, x):
        x_scaled = x * self.input_scale
        out = [x] if self.include_input else []

        angles = x_scaled.unsqueeze(-1) * self.freq_bands * np.pi
        sin_feats = torch.sin(angles).flatten(start_dim=1)
        cos_feats = torch.cos(angles).flatten(start_dim=1)

        out.append(sin_feats)
        out.append(cos_feats)
        return torch.cat(out, dim=-1)
    
print("Successfully defined Fourier Feature Encoding")

#### **5.1: Building the Model**

In [ ]:
class RelativeEdgeConv(MessagePassing):
    def __init__(self, hidden_dim, pos_dim=3):
        super().__init__(aggr='max')  # max aggregation, matching the original PointNetConv setup
        self.local_nn = Sequential(Linear(2 * hidden_dim + pos_dim + 1, hidden_dim), Tanh())

    def forward(self, h, pos, edge_index):
        return self.propagate(edge_index, h=h, pos=pos)

    def message(self, h_j, h_i, pos_j, pos_i):
        rel_h    = h_j - h_i
        rel_pos  = pos_j - pos_i
        rel_dist = rel_pos.norm(dim=-1, keepdim=True)
        return self.local_nn(torch.cat([h_j, rel_h, rel_pos, rel_dist], dim=-1))

class AttentionPool(nn.Module):
    def __init__(self, hidden_dim, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.score_fn = Sequential(Linear(hidden_dim, hidden_dim // 2), Tanh(), Linear(hidden_dim // 2, n_heads))

    def out_dim(self, hidden_dim):
        return hidden_dim * self.n_heads

    def forward(self, h, batch):
        scores = self.score_fn(h)   # [N, n_heads]
        outs = []
        for head in range(self.n_heads):
            alpha = pyg_softmax(scores[:, head], batch)                      # [N], softmax within each graph
            outs.append(global_add_pool(alpha.unsqueeze(-1) * h, batch))     # [B, hidden_dim]
        return torch.cat(outs, dim=1)   # [B, hidden_dim * n_heads]

class ShowerTrackGNN(torch.nn.Module):
    def __init__(self, hidden_dim=64, n_layers=3,
                 spatial_freqs=6, temporal_freqs=6, include_input=True,
                 spatial_scale=1.0, temporal_scale=1.0,
                 global_dim=len(GLOBAL_FEATS), dropout=DROPOUT_P, attn_heads=ATTN_HEADS):
        super().__init__()

        self.spatial_encoder = FourierFeatureEncoder(num_freqs=spatial_freqs, include_input=include_input, input_scale=spatial_scale)
        self.temporal_encoder = FourierFeatureEncoder(num_freqs=temporal_freqs, include_input=include_input, input_scale=temporal_scale)

        spatial_out_dim  = self.spatial_encoder.out_dim(3)   # X, Y, Z
        temporal_out_dim = self.temporal_encoder.out_dim(1)  # T0
        input_dim = spatial_out_dim + temporal_out_dim + 1   # +1 for raw E

        self.input_proj = Linear(input_dim, hidden_dim)
        self.pos_reinject = Linear(spatial_out_dim, hidden_dim)
        self.conv_layers = torch.nn.ModuleList([RelativeEdgeConv(hidden_dim) for _ in range(n_layers)])
        self.layer_norms = torch.nn.ModuleList([LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.attn_pool = AttentionPool(hidden_dim, n_heads=attn_heads)
        self.global_encoder = Sequential(Linear(global_dim, hidden_dim), nn.GELU(), Linear(hidden_dim, hidden_dim))
        pooled_dim = hidden_dim * 2 + self.attn_pool.out_dim(hidden_dim) + hidden_dim

        self.classifier = Sequential(
            Linear(pooled_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            Linear(hidden_dim, hidden_dim // 2),
            Tanh(),
            Linear(hidden_dim // 2, 1),
        )

    def forward(self, data):
        x, pos, batch = data.x, data.pos, data.batch
        edge_index = data.edge_index  # precomputed Delaunay graph (Section 4.1)

        xyz = x[:, 0:3]
        t0  = x[:, 3:4]
        e   = x[:, 4:5]

        xyz_enc = self.spatial_encoder(xyz)
        t0_enc  = self.temporal_encoder(t0)
        h = torch.cat([xyz_enc, t0_enc, e], dim=-1)
        h = torch.tanh(self.input_proj(h))

        # Relative-position input for RelativeEdgeConv -- spatial only (X, Y, Z).
        pos_xyz = pos[:, :3]

        # Fourier reinjection bias, computed once and reused at every layer.
        pos_bias = self.pos_reinject(xyz_enc)

        for conv, norm in zip(self.conv_layers, self.layer_norms):
            h_in = h
            h = h + pos_bias
            h = torch.tanh(norm(conv(h, pos_xyz, edge_index)))
            h = self.dropout(h)
            h = h + h_in

        h_mean = global_mean_pool(h, batch)
        h_max  = global_max_pool(h, batch)
        h_attn = self.attn_pool(h, batch)
        u_enc  = self.global_encoder(data.u)

        pooled = torch.cat([h_mean, h_max, h_attn, u_enc], dim=1)

        return self.classifier(pooled)


model     = ShowerTrackGNN(hidden_dim=128, n_layers=4, spatial_freqs=8, temporal_freqs=8, dropout=DROPOUT_P, attn_heads=ATTN_HEADS).to(device)
criterion = BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00023071869376920548, weight_decay=0.0015258427356571313)

scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr = 0.000650714943479, epochs = EPOCHS, steps_per_epoch = len(train_loader), pct_start = 0.08)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready — trainable parameters: {total_params:,}")
print(model)


#### **6.1: Training and Evaluation Loop**

In [ ]:
def run_epoch(loader, model, criterion, optimizer=None, scheduler=None, scheduler_step_per_batch=True):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds  = []
    all_labels = []

    with (torch.enable_grad() if is_train else torch.no_grad()):
        for batch in loader:
            batch = batch.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad()

            logits = model(batch).squeeze(-1)
            labels = batch.y.squeeze(-1)
            loss   = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()
                if scheduler is not None and scheduler_step_per_batch:
                    scheduler.step()

            total_loss += loss.item() * batch.num_graphs

            probs = torch.sigmoid(logits)
            preds = (probs > DECISION_THRESHOLD).long()
            all_preds.append(preds.cpu())
            all_labels.append(labels.long().cpu())

    mean_loss = total_loss / len(loader.dataset)
    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    accuracy   = (all_preds == all_labels).float().mean().item()

    return mean_loss, accuracy


#### **6.2: Hyperparameter Optimization**

In [ ]:
def build_trial_optimizer(trial, params):
    opt_name = params['optimizer_name']
    kwargs = dict(lr=params['lr'], weight_decay=params['weight_decay'])
    if opt_name == 'AdamW':
        return torch.optim.AdamW(params['model_params'], **kwargs)
    elif opt_name == 'Adam':
        return torch.optim.Adam(params['model_params'], **kwargs)
    else:  # 'RMSprop'
        return torch.optim.RMSprop(params['model_params'], **kwargs)


def build_trial_scheduler(trial, optimizer, params, steps_per_epoch):
    sched_name = params['scheduler_name']
    if sched_name == 'onecycle':
        # Per-batch scheduler -- run_epoch steps it internally every batch.
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=params['lr'] * params['max_lr_mult'], epochs=EPOCHS_PER_TRIAL, steps_per_epoch=steps_per_epoch)
        return scheduler, True   # step_per_batch=True
    elif sched_name == 'cosine':
        # Per-epoch scheduler -- caller steps it once after each epoch.
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PER_TRIAL, eta_min=params['lr'] * 0.01)
        return scheduler, False
    else:  # 'plateau'
        # Per-epoch scheduler, stepped with the val_loss metric -- caller
        # calls scheduler.step(val_loss) once after each epoch instead.
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=params['plateau_factor'],patience=params['plateau_patience'])
        return scheduler, False


def objective(trial):
    params = {
        # optimizer
        'optimizer_name': trial.suggest_categorical('optimizer_name', ['AdamW', 'Adam', 'RMSprop']),
        'lr':              trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'weight_decay':    trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
        # LR schedule
        'scheduler_name':  trial.suggest_categorical('scheduler_name', ['onecycle', 'cosine', 'plateau']),
        'max_lr_mult':     trial.suggest_float('max_lr_mult', 1.5, 10.0, log=True),
        'plateau_factor':  trial.suggest_float('plateau_factor', 0.1, 0.5),
        'plateau_patience':trial.suggest_int('plateau_patience', 2, 6),
        # architecture
        'hidden_dim':      trial.suggest_categorical('hidden_dim', [32, 64, 96, 128]),
        'n_layers':        trial.suggest_int('n_layers', 2, 4),
        'dropout':         trial.suggest_float('dropout', 0.0, 0.4),
        'attn_heads':      trial.suggest_categorical('attn_heads', [2, 4, 8]),
    }

    trial_model = ShowerTrackGNN(hidden_dim=params['hidden_dim'], n_layers=params['n_layers'], spatial_freqs=6, temporal_freqs=6,
                                 dropout=params['dropout'], attn_heads=params['attn_heads'],).to(device)
    trial_criterion = BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

    params['model_params'] = trial_model.parameters()
    trial_optimizer = build_trial_optimizer(trial, params)
    trial_scheduler, step_per_batch = build_trial_scheduler(trial, trial_optimizer, params, steps_per_epoch=len(train_loader))

    best_trial_val_loss = float('inf')
    best_trial_val_acc  = -float('inf')

    for epoch in range(1, EPOCHS_PER_TRIAL + 1):
        run_epoch(train_loader, trial_model, trial_criterion, trial_optimizer,
                  scheduler=trial_scheduler if step_per_batch else None,scheduler_step_per_batch=step_per_batch,)
        val_loss, val_acc = run_epoch(val_loader, trial_model, trial_criterion)

        if not step_per_batch:
            if params['scheduler_name'] == 'plateau':
                trial_scheduler.step(val_loss)
            else:  # cosine
                trial_scheduler.step()

        best_trial_val_loss = min(best_trial_val_loss, val_loss)
        best_trial_val_acc  = max(best_trial_val_acc, val_acc)

        report_value = val_loss if OPTIMIZE_METRIC == 'loss' else (1.0 - val_acc)
        trial.report(report_value, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_trial_val_loss if OPTIMIZE_METRIC == 'loss' else (1.0 - best_trial_val_acc)


if RUN_OPTIMIZATION:
    # run actual optimization cycle
    pbar = tqdm(total=N_OPTUNA_TRIALS, desc="Optuna Trials")

    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42), pruner=MedianPruner(n_warmup_steps=3))
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[lambda study, trial: pbar.update(1)])
    pbar.close()

    print(f"\nBest trial: #{study.best_trial.number}")
    print(f"Best {OPTIMIZE_METRIC} objective: {study.best_value:.4f}")
    print("Best params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")

    # save best result to CSV log
    log_file = Path("optuna_log.csv")
    fieldnames = ["trial", "metric", "objective", *study.best_params.keys()]
    write_header = not log_file.exists()

    with log_file.open("a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow({"trial": study.best_trial.number, "metric": OPTIMIZE_METRIC, "objective": study.best_value, **study.best_params})

#### **7.1: Train Model**

In [ ]:
CHECKPOINT_DIR.mkdir(exist_ok=True, parents=True)
best_val_loss = float('inf')
best_epoch = -1
history = []
epochs_since_improvement = 0

print(f"\nStarting training — {EPOCHS} epochs, batch size {BATCH_SIZE}")
print(f"Logging to {LOG_PATH}\n")

with open(LOG_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f,fieldnames=['epoch', 'train_loss', 'val_loss', 'train_acc', 'val_acc'])
    writer.writeheader()

epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Training", unit="epoch")

for epoch in epoch_bar:
    train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer, scheduler)
    val_loss, val_acc = run_epoch(val_loader, model, criterion)
    row = {'epoch': epoch,'train_loss': round(train_loss, 6),'val_loss': round(val_loss, 6),'train_acc': round(train_acc, 6),'val_acc': round(val_acc, 6),}
    history.append(row)

    with open(LOG_PATH, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=row.keys()).writerow(row)

    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_since_improvement = 0
        torch.save({'epoch': epoch,'model_state_dict': model.state_dict(),'optimizer_state_dict': optimizer.state_dict(),
                    'val_loss': val_loss,'val_acc': val_acc,'max_coord': MAX_COORD,'means': means,'stds': stds,'decision_threshold': DECISION_THRESHOLD,},
                   CHECKPOINT_DIR / 'best_model.pt')
    else:
        epochs_since_improvement += 1

    epoch_bar.set_postfix(train_loss=f"{train_loss:.4f}",val_loss=f"{val_loss:.4f}",val_acc=f"{val_acc:.4f}",best=best_epoch,patience=f"{epochs_since_improvement}/{PATIENCE}",)

    if epochs_since_improvement >= PATIENCE:
        print(f"\nNo improvement in {PATIENCE} epochs — stopping early at epoch {epoch}.")
        break

print(f"\nTraining complete. Best checkpoint: epoch {best_epoch} (val_loss={best_val_loss:.4f})")

#### **8.1: Analyze Performance**

In [ ]:
PLOTS_PATH.mkdir(parents=True, exist_ok=True)
checkpoint = torch.load(CHECKPOINT_DIR / 'best_model.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model from epoch {checkpoint['epoch']} "
      f"(val_loss={checkpoint['val_loss']:.4f})")

all_probs  = []
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch  = batch.to(device)
        logits = model(batch).squeeze(-1)
        probs  = torch.sigmoid(logits)
        preds  = (probs > DECISION_THRESHOLD).long()

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(batch.y.squeeze(-1).long().cpu().numpy())

all_probs  = np.concatenate(all_probs)
all_preds  = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

acc               = accuracy_score(all_labels, all_preds)
prec, rec, f1, _  = precision_recall_fscore_support(
    all_labels, all_preds, average='binary', pos_label=1
)
auc = roc_auc_score(all_labels, all_probs)

print(f"\n Test Set Metrics")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}  (of predicted tracks, how many are really tracks)")
print(f"Recall:    {rec:.4f}  (of all true tracks, how many did we catch)")
print(f"F1:        {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")

# ROC curve graph
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, lw=2, label=f'GNN (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Shower vs Track')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_PATH / 'roc_curve.png', dpi=150)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
print("Saved roc_curve.png")


# confusion matrix graph
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, matrix, title, fmt in zip(
    axes,
    [cm,      cm_norm],
    ['Confusion Matrix (counts)', 'Confusion Matrix (normalized)'],
    ['d',     '.2%'],
):
    sns.heatmap(
        matrix, annot=True, fmt=fmt, cmap='Blues', ax=ax,
        xticklabels=['Shower', 'Track'],
        yticklabels=['Shower', 'Track'],
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)

plt.tight_layout()
plt.savefig(PLOTS_PATH / 'confusion_matrix.png', dpi=150)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
print("Saved confusion_matrix.png")

# training curves
log_df = pd.read_csv(LOG_PATH)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(log_df['epoch'], log_df['train_loss'], label='Train')
ax1.plot(log_df['epoch'], log_df['val_loss'],   label='Val')
ax1.axvline(best_epoch, color='red', linestyle='--', alpha=0.6,
            label=f'Best epoch ({best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss over Training')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(log_df['epoch'], log_df['train_acc'], label='Train')
ax2.plot(log_df['epoch'], log_df['val_acc'],   label='Val')
ax2.axvline(best_epoch, color='red', linestyle='--', alpha=0.6,
            label=f'Best epoch ({best_epoch})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy over Training')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_PATH / 'training_curves.png', dpi=150)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
print("Saved training_curves.png")

# score distribution histogram
shower_mask = all_labels == 0
track_mask  = all_labels == 1
n_test_total = len(all_labels)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(all_probs[shower_mask], bins=60, alpha=0.6, label='True Shower',
        color='steelblue', weights=np.full(shower_mask.sum(), 100.0 / n_test_total))
ax.hist(all_probs[track_mask],  bins=60, alpha=0.6, label='True Track',
        color='darkorange', weights=np.full(track_mask.sum(), 100.0 / n_test_total))
ax.axvline(DECISION_THRESHOLD, color='red', linestyle='--',
           label=f'Decision boundary ({DECISION_THRESHOLD})')
ax.set_xlabel('Predicted probability of being a Track')
ax.set_ylabel('% of test set')
ax.set_title('Score Distribution by True Class')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_PATH / 'score_distribution.png', dpi=150)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
print("Saved score_distribution.png")


#### **8.2: Score Distribution Behavior**

In [ ]:
test_clusters = [clusters[i] for i in test_idx]
test_pdgs = np.array([c['pdg'] for c in test_clusters])
assert len(test_clusters) == len(all_probs) == len(all_labels)
test_n_hits = np.array([c['n_hits'] for c in test_clusters])

category_masks = {
    name: np.isin(test_pdgs, list(pdg_set))
    for name, pdg_set in PDG_CATEGORIES.items()
}

# other PDGs that are not in the listed category are labeled as "other"
other_mask = ~np.logical_or.reduce(list(category_masks.values()))

shower_mask = (all_labels == 0)
track_mask  = (all_labels == 1)

# sanity check: electron + photon should reconstruct true shower exactly, and the others should match true track exactly
shower_from_species = category_masks['Electron'] | category_masks['Photon']
track_from_species = (category_masks['Muon'] | category_masks['Proton'] | category_masks['Pion'] | category_masks['Kaon'] | other_mask)
shower_match = np.array_equal(shower_from_species, shower_mask)
track_match = np.array_equal(track_from_species, track_mask)
print(f"Electron+Photon == True Shower: {shower_match} ({shower_from_species.sum():,} vs {shower_mask.sum():,})")
print(f"Muon+Proton+Pion+Kaon+Other == True Track:       {track_match} ({track_from_species.sum():,} vs {track_mask.sum():,})")
assert shower_match and track_match, "Species grouping does not reconcile with shower/track labels -- check PDG_CATEGORIES"

fig, ax = plt.subplots(figsize=(9, 6))
n_total = len(all_probs)

def frac_weights(mask):
    return np.full(mask.sum(), 100.0 / n_total)


ax.hist(all_probs[shower_mask], bins=60, range=(0, 1), alpha=0.4, weights=frac_weights(shower_mask), label='True Shower', color='steelblue', histtype='stepfilled')
ax.hist(all_probs[track_mask], bins=60, range=(0, 1), alpha=0.4, weights=frac_weights(track_mask), label='True Track', color='darkorange', histtype='stepfilled')

for name, mask in category_masks.items():
    n = mask.sum()
    if n == 0:
        continue
    ax.hist(all_probs[mask], bins=60, range=(0, 1), weights=frac_weights(mask), label=f'{name} (n={n:,})', color=CATEGORY_COLORS[name], histtype='step', linewidth=1.8)

if other_mask.sum() > 0:
    ax.hist(all_probs[other_mask], bins=60, range=(0, 1), weights=frac_weights(other_mask), label=f'Other (n={other_mask.sum():,})', color='gray', histtype='step', linewidth=1.8, linestyle=':')
ax.axvline(0.5, color='red', linestyle='--', label='Decision boundary (0.5)')
ax.set_yscale('log')
ax.set_xlabel('Predicted probability of being a Track')
ax.set_ylabel('% of test set (log scale)')
ax.set_title('Score Distribution by True Particle Species (mcPDG)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_PATH / 'score_distribution_by_species_log.png', dpi=150)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
print("Saved score_distribution_by_species_log.png")


# score distribution vs size of a cluster (for true showers)
fig1, ax1 = plt.subplots(figsize=(7, 6))
hb1 = ax1.hexbin(test_n_hits[shower_mask], all_probs[shower_mask],
                  gridsize=40, xscale='log', cmap='viridis',
                  norm=LogNorm(), mincnt=1)
ax1.set_xlabel('Hits per cluster (log scale)')
ax1.set_ylabel('Predicted P(track)')
ax1.set_title('Confidence vs. hit count — true Showers')
fig1.colorbar(hb1, ax=ax1, label='cluster count')
plt.tight_layout()
plt.savefig(PLOTS_PATH / 'explore_confidence_vs_hitcount_showers.png', dpi=150)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig1)
print("Saved explore_confidence_vs_hitcount_showers.png")


# score distribution vs size of a cluster (for true tracks)
fig2, ax2 = plt.subplots(figsize=(7, 6))
hb2 = ax2.hexbin(test_n_hits[track_mask], all_probs[track_mask],
                  gridsize=40, xscale='log', cmap='viridis',
                  norm=LogNorm(), mincnt=1)
ax2.set_xlabel('Hits per cluster (log scale)')
ax2.set_ylabel('Predicted P(track)')
ax2.set_title('Confidence vs. hit count — true Tracks')
fig2.colorbar(hb2, ax=ax2, label='cluster count')
plt.tight_layout()
plt.savefig(PLOTS_PATH / 'explore_confidence_vs_hitcount_tracks.png', dpi=150)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig2)
print("Saved explore_confidence_vs_hitcount_tracks.png")

#### **8.3 Accuracy Measurements**

In [ ]:
def accuracy_vs_metric_by_species(metric_values, metric_name, preds, labels, pdgs, pdg_categories=PDG_CATEGORIES, colors=CATEGORY_COLORS,
                                   n_bins=12, min_per_bin=20, log_x=True, save_path=None, show=True):
    correct = (preds == labels)

    if log_x:
        bin_edges = np.logspace(np.log10(metric_values.min()), np.log10(metric_values.max()), n_bins + 1)
        bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])  # geometric mean matches log spacing
    else:
        bin_edges = np.linspace(metric_values.min(), metric_values.max(), n_bins + 1)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    category_masks = {name: np.isin(pdgs, list(pdg_set)) for name, pdg_set in pdg_categories.items()}
    other_mask = ~np.logical_or.reduce(list(category_masks.values())) if category_masks else np.ones_like(pdgs, dtype=bool)
    all_masks = {**category_masks, 'Other': other_mask}
    all_colors = {**colors, 'Other': 'gray'}

    fig, ax = plt.subplots(figsize=(9, 6))
    for name, species_mask in all_masks.items():
        accs = []
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
            bin_mask = species_mask & (metric_values >= lo) & (metric_values < hi)
            n = bin_mask.sum()
            accs.append(correct[bin_mask].mean() if n >= min_per_bin else np.nan)
        accs = np.array(accs)
        if np.all(np.isnan(accs)):
            continue
        ax.plot(bin_centers, accs, marker='o', label=f'{name} (n={species_mask.sum():,})',
                color=all_colors.get(name))

    if log_x:
        ax.set_xscale('log')
    ax.set_xlabel(metric_name)
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1.02)
    ax.set_title(f'Classification Accuracy vs. {metric_name}, by True Particle Species')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig, ax


# Accuracy vs. number of hits
fig, ax = accuracy_vs_metric_by_species(metric_values=test_n_hits, metric_name='Number of hits in cluster',preds=all_preds, labels=all_labels,
                                        pdgs=test_pdgs, log_x=True,save_path=PLOTS_PATH / 'accuracy_vs_hits_by_species.png', show=SHOW_PLOTS,)
print("Saved accuracy_vs_hits_by_species.png")

# Accuracy vs. energy
test_energy = np.array([c['energy'] for c in test_clusters])
fig, ax = accuracy_vs_metric_by_species(metric_values=test_energy, metric_name='Energy',preds=all_preds, labels=all_labels, pdgs=test_pdgs, log_x=True,
    save_path=PLOTS_PATH / 'accuracy_vs_energy_by_species.png', show=SHOW_PLOTS,)
print("Saved accuracy_vs_energy_by_species.png")


#### **9.1: Feature Importance**

In [ ]:
def node_feature_saliency(model, loader, max_batches=20):
    # fourier-feature encoded path and relativeEdgeConv term- gradients are summed to capture the features influence on output
    model.eval()
    names = ['X', 'Y', 'Z', 'T0', 'E']
    grad_sums = torch.zeros(5, device=device)
    n_nodes_seen = 0

    for i, batch in enumerate(loader):
        if i >= max_batches:
            break
        batch = batch.to(device)
        batch.x = batch.x.detach().clone().requires_grad_(True)
        batch.pos = batch.pos.detach().clone().requires_grad_(True)

        logits = model(batch).squeeze(-1)
        logits.sum().backward()

        g_x = batch.x.grad.abs().sum(dim=0)      # [X, Y, Z, T0, E]
        g_pos = batch.pos.grad.abs().sum(dim=0)  # [X, Y, Z, T0]

        grad_sums[:4] += g_x[:4] + g_pos[:4]
        grad_sums[4] += g_x[4]  # E only appears in x, not in pos
        n_nodes_seen += batch.x.shape[0]

    avg = (grad_sums / n_nodes_seen).cpu().numpy()
    return dict(zip(names, avg))


node_importance = node_feature_saliency(model, test_loader)

print("Node feature importance (mean |d(logit)/d(feature)|, feature path + geometric path):")
for k, v in sorted(node_importance.items(), key=lambda kv: -kv[1]):
    print(f"  {k:>4s}: {v:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
names_sorted = sorted(node_importance, key=lambda k: -node_importance[k])
ax.bar(names_sorted, [node_importance[n] for n in names_sorted], color='darkorange')
ax.set_ylabel('Mean |d(logit) / d(feature)|')
ax.set_title('Node-level Feature Importance -- Input-Gradient Saliency')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(PLOTS_PATH / 'node_feature_importance.png', dpi=150)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
print("Saved node_feature_importance.png")





def global_feature_permutation_importance(model, loader, feature_names=GLOBAL_FEATS, n_repeats=3, metric='auc'):
    model.eval()
    # cluster level feature permutations are shuffled one by one and for each shuffle, the drop in AUC is recorded
    def score(perm_col=None):
        all_probs, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(device)
                if perm_col is not None:
                    u = batch.u.clone()
                    perm = torch.randperm(u.shape[0], device=device)
                    u[:, perm_col] = u[perm, perm_col]
                    batch.u = u
                logits = model(batch).squeeze(-1)
                all_probs.append(torch.sigmoid(logits).cpu().numpy())
                all_labels.append(batch.y.squeeze(-1).cpu().numpy())
        probs = np.concatenate(all_probs)
        labs = np.concatenate(all_labels)
        if metric == 'auc':
            return roc_auc_score(labs, probs)
        return accuracy_score(labs, (probs > DECISION_THRESHOLD).astype(int))

    baseline = score(None)
    importances = {}
    for j, name in enumerate(feature_names):
        drops = [baseline - score(j) for _ in range(n_repeats)]
        importances[name] = (float(np.mean(drops)), float(np.std(drops)))

    print(f"Baseline {metric.upper()}: {baseline:.4f}\n")
    print(f"Permutation importance ({metric.upper()} drop when shuffled, mean +/- std over {n_repeats} repeats):")
    for name, (mean_drop, std_drop) in sorted(importances.items(), key=lambda kv: -kv[1][0]):
        print(f"  {name:>12s}: {mean_drop:+.4f} +/- {std_drop:.4f}")

    return baseline, importances


global_baseline, global_importances = global_feature_permutation_importance(model, test_loader, metric='auc')

fig, ax = plt.subplots(figsize=(9, 6))
names_sorted = sorted(global_importances, key=lambda k: -global_importances[k][0])
means = [global_importances[n][0] for n in names_sorted]
stds  = [global_importances[n][1] for n in names_sorted]
ax.barh(names_sorted[::-1], means[::-1], xerr=stds[::-1], color='steelblue')
ax.set_xlabel(f'Drop in AUC when feature is shuffled (baseline AUC = {global_baseline:.4f})')
ax.set_title('Global (cluster-level) Feature Importance -- Permutation Test')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(PLOTS_PATH / 'global_feature_importance.png', dpi=150)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
print("Saved global_feature_importance.png")

